In [ ]:
%matplotlib inline

# Inlet Profiles

Chromatographic systems always require some kind of convective flow through the column.

In this lesson, we will:
- Set up the Inlet, Outlet, and Flow Sheet manually.
- Learn about defining events.
- Define inlet profiles using piecewise cubic polynomials.
- Derive inlet profiles from mock experimental data.

## Example 1: Flow from `Inlet` to `Outlet`

In this example, we will examine a simple system with two unit operations: an [Inlet](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.Inlet.html) and an [Outlet](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.Outlet.html).

```{figure} ./resources/IO.png
:width: 30%
```

We will introduce flow from the `Inlet` to the `Outlet` with a constant flow rate of $Q = 1~mL \cdot s^{-1}$.
Initially, the concentration is $1.0~mM$. After $1~min$, it changes to $0.0~mM$.

```{figure} ./resources/step.png
:width: 30%
```

## 1. Setting up the model

Start with the `ComponentSystem`:

## Inlet

The `Inlet` pseudo unit operation serves as the source for the system.
It creates arbitrary concentration profiles as boundary conditions (see also [here](https://cadet-process.readthedocs.io/en/latest/reference/generated/CADETProcess.processModel.Inlet.html#CADETProcess.processModel.Inlet)).

Note, both concentration `c` and `flow_rate` are `section_dependent_parameters` which can later be modified by `Events`.
Here, we explicitly fix the flow rate.

## Outlet

The outlet setup is straightforward:

## Flow Sheet Connectivity

Add the unit operations to a `FlowSheet`:

## Dynamic Events in Process

Dynamic changes of model parameters or flow sheet connections are configured in the `Process` class.
For more information, see [here](https://cadet-process.readthedocs.io/en/latest/user_guide/process_model/process.html).

To add an event that changes the value of a parameter, use the `add_event` method.
It requires the following arguments:
- `name`: Name of the event.
- `parameter_path`: Path of the parameter that is changed in dot notation.
  For example, the flow rate of the eluent unit is the parameter `flow_rate` of the `eluent` unit in the `flow_sheet`.
  Hence, the path is `flow_sheet.eluent.flow_rate`.
  As previously mentioned, the name of the unit operation is used to reference it, not the variable.
- `state`: Value of the attribute that is changed at event execution.
- `time`: Time at which the event is executed.

To display all time dependent parameters of an object, inspect the `section_dependent_parameters` attribute.

Note that also flow sheet connectivity can be added as events.
More on that later.

All events are stored in the events attribute.
To visualize the trajectory of the parameter state over the entire cycle, the Process provides a `plot_events()` method.

## 3. Setting up the simulator and running the simulation

To simulate the process, configure a process simulator.
If no path is specified, CADET-Process will attempt to autodetect CADET.

Now, run the simulation:

## 4. Plotting the results

The `simulation_results` object contains the solution for the inlet and outlet of every unit operation.
It also provides plot methods.

## 5. Other inlet profiles

Processes may require more complex inlet profiles.
CADET-Process allows us to add concentration gradients as high-degree polynomials.

To create a linear concentration gradient from $y_0$ to $y_1$ over a duration $T$, use the following first-degree polynomial:

$$
y(t) = y_0 + \frac{y_1 - y_0}{T} \cdot t \quad \text{for} \quad 0 \le t \le T
$$

For example, to achieve a gradient from 0 to 1 over 60 seconds, the polynomial becomes:

$$
y(t) = 0 + \frac{1 - 0}{60} \cdot t = \frac{t}{60}
$$

The polynomial coefficients are:
$$
a_0 = 0, \quad a_1 = \frac{1}{60}
$$

Since we are still working with the same process object, if we want to replace an event, we need to remove the old one first.
Event names must always be unique.

Then we add the new flow rate as a polynomial based on the flow rate we calculated:

When the inlet profile cannot be represented by a polynomial, CADET-Process offers the functionality to add inlet profiles from arrays.
First, let's create a complex simulated inlet profile with some noise:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def noisy_inlet(t, seed=0):
    r = np.random.default_rng(seed)
    s = np.exp(-0.5*((t-30)/5)**2) + 0.7*np.exp(-0.5*((t-80)/9)**2)
    d = 0.03*np.sin(2*np.pi*0.02*t)
    n = r.normal(0, 0.02, t.size)*10
    k = np.exp(-np.abs(np.arange(-15, 16))/5)
    y = np.convolve(n, k, 'same')/k.sum()
    return np.clip(s + d + y, 0, None)

t = np.linspace(0, 120, 100)
c = noisy_inlet(t).reshape(-1,1)

plt.plot(t,c)

To add it, use `add_concentration_profile` method.
This method requires the target unit operation, as well as the time and concentration of the inlet profile.